<a href="https://colab.research.google.com/github/cc-huang-0716/internship_tests/blob/main/hilbert_huang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import scipy
from scipy.signal import find_peaks, hilbert
from scipy.interpolate import CubicSpline, PchipInterpolator
import google.colab
from google.colab import drive

# 數據加載
drive.mount("/content/drive")
anomaly_data = pd.read_csv("/content/drive/MyDrive/peak_series_data.csv")
stable_data = pd.read_csv("/content/drive/MyDrive/stable_series_500.csv")
df1 = pd.DataFrame(anomaly_data)
df2 = pd.DataFrame(stable_data)
merge_df = pd.concat([df1, df2], axis = 1)
print("數據加載成功")

# 極值與包絡
def find_extrema_indices(y, distance=1):
    """
    回傳 (max_idx, min_idx)，距離參數可抑制雜訊尖峰（>=1）。
    """
    y = np.asarray(y, float)
    # 極大：find_peaks(y)，極小：find_peaks(-y)
    max_idx, _ = find_peaks(y, distance=distance)
    min_idx, _ = find_peaks(-y, distance=distance)
    return max_idx, min_idx

def build_envelope(x, y, idx, method="pchip"):
    """
    用極值節點建立包絡；若節點不足兩個 → 用端點線性退化。
    x: 單調遞增（等/不等間距皆可）
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    idx = np.asarray(idx, int)
    if len(idx) >= 2:
        Xk = x[idx]; Yk = y[idx]
        if method == "spline":
            itp = CubicSpline(Xk, Yk, bc_type="natural", extrapolate=True)
        else:
            itp = PchipInterpolator(Xk, Yk, extrapolate=True)
        return itp(x)
    else:
        # 不足以插值：用端點直線
        return np.interp(x, [x[0], x[-1]], [y[0], y[-1]])

# 單次 sifting

def _zero_crossings(v):
    s = np.sign(v).astype(int)
    s[s == 0] = 1
    return int(np.sum(s[:-1] * s[1:] < 0))

def sift_once(y, x=None, distance=1, env_method="pchip",
              eps_mean=0.05, sd_tol=0.25, max_iter=50):
    """
    對 y 做一次 sifting，回傳 (imf, Eu, El, m, iters)。
    eps_mean: ||m||/||h|| 門檻（0.05~0.1 常用）
    sd_tol  : SD 準則門檻（0.2~0.3 常用）
    """
    y = np.asarray(y, float)
    if x is None:
        x = np.arange(len(y), dtype=float)
    else:
        x = np.asarray(x, float)

    h = y.copy()
    h_prev = None

    for it in range(1, max_iter + 1):
        max_idx, min_idx = find_extrema_indices(h, distance=distance)

        Eu = build_envelope(x, h, max_idx, method=env_method)
        El = build_envelope(x, h, min_idx, method=env_method)
        m  = 0.5 * (Eu + El)
        h  = h - m

        # IMF 條件：零交叉與極值數差 <= 1，且均值相對幅度小
        ex = len(max_idx) + len(min_idx)
        zc = _zero_crossings(h)
        mean_small = (np.linalg.norm(m) / (np.linalg.norm(h) + 1e-20)) < eps_mean
        imf_like   = (abs(ex - zc) <= 1) and mean_small

        # SD 準則
        sd_ok = False
        if h_prev is not None:
            sd = np.sum((h_prev - h)**2) / (np.sum(h_prev**2) + 1e-20)
            sd_ok = (sd < sd_tol)
        h_prev = h.copy()

        if imf_like or sd_ok:
            return h, Eu, El, m, it

    # 到了上限也先交付
    return h, Eu, El, m, it

# EMD 外層：疊 IMF，直到殘差單調

def emd(y, x, max_imfs, distance, env_method,
        eps_mean, sd_tol, max_iter):
    """
    回傳 (IMFs[ k x N ], residue[N], meta_list)
    meta_list[k] 內含 Eu/El/m/iters，可做除錯或視覺化。
    """
    y = np.asarray(y, float)
    if x is None:
        x = np.arange(len(y), dtype=float)
    else:
        x = np.asarray(x, float)

    r = y.copy()
    imfs = []
    meta = []

    for _ in range(max_imfs):
        max_idx, min_idx = find_extrema_indices(r, distance=distance)
        if (len(max_idx) + len(min_idx)) < 2:  # 殘差單調或幾乎單調
            break

        imf, Eu, El, m, iters = sift_once(
            r, x=x, distance=distance, env_method=env_method,
            eps_mean=eps_mean, sd_tol=sd_tol, max_iter=max_iter
        )
        imfs.append(imf)
        meta.append({"Eu": Eu, "El": El, "m": m, "iters": iters})
        r = r - imf

    IMFs = np.vstack(imfs) if imfs else np.empty((0, len(y)))
    return IMFs, r, meta

# 希爾伯特轉換與瞬時量

def hilbert_all(IMFs, x):
    """
    對每個 IMF 計算瞬時幅度 A、相位 phi、瞬時頻率 f。
    """
    IMFs = np.asarray(IMFs, float)
    if IMFs.ndim == 1:
        IMFs = IMFs[None, :]
    N = IMFs.shape[1]

    if x is None:
        x = np.arange(N, dtype=float)
    else:
        x = np.asarray(x, float)

    out = []
    for k in range(IMFs.shape[0]):
        z = hilbert(IMFs[k])              # 解析訊號：實部=IMF，虛部=Hilbert 變換
        A = np.abs(z)                     # 瞬時幅度
        phi = np.unwrap(np.angle(z))      # 相位
        # 不等間距：用 np.gradient(phi, x)；等間距退化為 / dx
        f = np.gradient(phi, x) / (2*np.pi)
        out.append({"A": A, "phi": phi, "f": f, "z": z})
    return out
"""
A : 瞬時幅度，表數列幅度
phi : 瞬時相位，角度的累積量，需做處理
f : 瞬時頻率，表數列節奏，常有 noise 或 NaN
z : 複數解析信號，包含上述東西，不直接進分析
"""

hilb_list = []
for idx, row in merge_df.iterrows():
    row_values = np.asarray(row, dtype=float)
    IMFs, residue, meta = emd(
    row_values, x=None,
    max_imfs=8,
    distance=1,
    env_method="pchip",
    eps_mean=0.05,
    sd_tol=0.25,
    max_iter=50
)
    hilb = hilbert_all(IMFs, x=np.arange(IMFs.shape[1])) # hilb[k]['A'], hilb[k]['f'] 可作 Hilbert spectrum / 異常偵測
    hilb_list.append(hilb)
    #print(hilb)

In [ ]:
sample_df_list = []
for i in hilb_list:
  sample_df = pd.DataFrame(i)
  sample_df_list.append(sample_df)




#hilb_list_df = pd.DataFrame(hilb_list)
#hilb_list_df.head()
#hilb_list_df.fillna(0, inplace=True)
#print(hilb_list_df)
#hilb_list_df.drop(column = [""])